Import libraries and connect to Neo4j

In [1]:
from pathlib import Path
from neo4j import GraphDatabase
from dotenv import load_dotenv
import pandas as pd
import os

In [2]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

load_dotenv(project_root / ".env")

URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USER")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(
    URI,
    auth=(USER, PASSWORD)
)

driver.verify_connectivity()

print("Connected to Neo4j successfully!")

Connected to Neo4j successfully!


Load extracted CSV files

In [3]:
data_dir = project_root / "data" / "extracted"

entities_df = pd.read_csv(
    data_dir / "entities.csv",
    encoding="utf-8-sig"
)

relationships_df = pd.read_csv(
    data_dir / "relationships.csv",
    encoding="utf-8-sig"
)

print("Entities:", len(entities_df))
print("Relationships:", len(relationships_df))

display(entities_df.head())
display(relationships_df.head())

Entities: 14
Relationships: 13


,id,name_si,label
0,D001,ඇල්ල,Destination
1,DI001,බදුල්ල,District
2,P001,ඌව පළාත,Province
3,A001,පුංචි ශ්‍රී පාදය,Attraction
4,A002,ඇල්ල රොක්,Attraction


,source,relation,target
0,D001,LOCATED_IN,DI001
1,DI001,PART_OF,P001
2,D001,HAS_ATTRACTION,A001
3,D001,HAS_ATTRACTION,A002
4,D001,HAS_ATTRACTION,A003


Create a unique ID constraint
add a common Entity label to every entity, alongside its specific type such as Destination or Attraction

In [4]:
driver.execute_query(
    """
    CREATE CONSTRAINT entity_id_unique IF NOT EXISTS
    FOR (n:Entity)
    REQUIRE n.id IS UNIQUE
    """,
    database_=DATABASE
)

print("Entity ID constraint created!")

Entity ID constraint created!


Insert all 14 entities

In [5]:
allowed_labels = {
    "Destination",
    "District",
    "Province",
    "Attraction",
    "Activity"
}

for row in entities_df.to_dict("records"):
    label = str(row["label"])

    if label not in allowed_labels:
        raise ValueError(f"Unexpected label: {label}")

    query = f"""
    MERGE (n:Entity {{id: $id}})
    SET n:{label}
    SET n.name_si = $name_si
    RETURN n
    """

    driver.execute_query(
        query,
        id=str(row["id"]),
        name_si=str(row["name_si"]),
        database_=DATABASE
    )

print("All entities inserted successfully!")

All entities inserted successfully!


Insert all 13 relationships

In [6]:
allowed_relations = {
    "LOCATED_IN",
    "PART_OF",
    "HAS_ATTRACTION",
    "HAS_ACTIVITY"
}

for row in relationships_df.to_dict("records"):
    relation = str(row["relation"])

    if relation not in allowed_relations:
        raise ValueError(f"Unexpected relationship: {relation}")

    query = f"""
    MATCH (a:Entity {{id: $source}})
    MATCH (b:Entity {{id: $target}})
    MERGE (a)-[r:{relation}]->(b)
    RETURN r
    """

    records, _, _ = driver.execute_query(
        query,
        source=str(row["source"]),
        target=str(row["target"]),
        database_=DATABASE
    )

    if not records:
        raise ValueError(
            f"Could not connect {row['source']} to {row['target']}"
        )

print("All relationships inserted successfully!")

All relationships inserted successfully!


Verify the graph

In [7]:
records, _, _ = driver.execute_query(
    """
    MATCH (n:Entity)
    RETURN count(n) AS total_nodes
    """,
    database_=DATABASE
)

print("Total entities:", records[0]["total_nodes"])

records, _, _ = driver.execute_query(
    """
    MATCH (:Entity)-[r]->(:Entity)
    RETURN count(r) AS total_relationships
    """,
    database_=DATABASE
)

print("Total relationships:", records[0]["total_relationships"])

Total entities: 14
Total relationships: 13


graph enrichment

Prepare detailed properties

In [9]:
enrichment_data = [
    {
        "id": "D001",
        "properties": {
            "elevation_m": 1041,
            "climate_si": "සාමාන්‍යයෙන් සිසිල් කඳුකර දේශගුණය",
            "source_section": "1, 2"
        }
    },
    {
        "id": "A001",
        "properties": {
            "elevation_m": 1141,
            "hiking_difficulty": "easy",
            "best_time_si": "උදෑසන සහ සවස් කාලය",
            "source_section": "3"
        }
    },
    {
        "id": "A002",
        "properties": {
            "hiking_difficulty": "challenging",
            "description_si": "පුංචි ශ්‍රී පාදයට සාපේක්ෂව වඩාත් අභියෝගාත්මක කඳු නැගීමකි.",
            "source_section": "4"
        }
    },
    {
        "id": "A004",
        "properties": {
            "height_m": 25,
            "distance_from_ella_km": 6,
            "source_section": "6"
        }
    },
    {
        "id": "A007",
        "properties": {
            "height_m": 220,
            "source_section": "9"
        }
    }
]

print("Entities prepared:", len(enrichment_data))

Entities prepared: 5


Update the existing nodes

In [10]:
for item in enrichment_data:
    records, _, _ = driver.execute_query(
        """
        MATCH (n:Entity {id: $id})
        SET n += $properties
        SET n.source_document = "ella.txt"
        RETURN n.id AS id
        """,
        id=item["id"],
        properties=item["properties"],
        database_=DATABASE
    )

    if not records:
        raise ValueError(
            f"Entity not found: {item['id']}"
        )

print("Node properties updated successfully!")

Node properties updated successfully!


Verify the properties

In [11]:
records, _, _ = driver.execute_query(
    """
    MATCH (n:Entity)
    WHERE n.id IN ["A001", "A002", "A004", "A007"]

    RETURN
        n.name_si AS name,
        n.height_m AS height,
        n.elevation_m AS elevation,
        n.hiking_difficulty AS difficulty

    ORDER BY name
    """,
    database_=DATABASE
)

for record in records:
    print(dict(record))

{'name': 'ඇල්ල රොක්', 'height': None, 'elevation': None, 'difficulty': 'challenging'}
{'name': 'දියලුම දිය ඇල්ල', 'height': 220, 'elevation': None, 'difficulty': None}
{'name': 'පුංචි ශ්\u200dරී පාදය', 'height': None, 'elevation': 1141, 'difficulty': 'easy'}
{'name': 'රාවණා ඇල්ල', 'height': 25, 'elevation': None, 'difficulty': None}
